# Sales RAG Lab — Structured Data: the Hybrid Pattern (S3 Vectors + Text-to-SQL)

Answers questions about a fictional **Hawaiian electronics-components retailer** (Arduino boards,
sensors, MOSFETs, marine electronics, …) across island branches. The sales data is **structured**
(per-branch CSV exports), which needs **two** answer paths — and a production agent needs *both*:

| Path | Good at | Bad at |
|---|---|---|
| **A. Serialize rows → embed → S3 Vectors** | fuzzy, qualitative lookups | exact totals / counts / ranking |
| **B. Text-to-SQL over Athena** | precise aggregation | open-ended semantic questions |

The deployed `Sales_Specialist` agent routes each question to the right tool. Getting this wrong is
the #1 way structured-data RAG silently returns confidently-wrong numbers.

**This notebook is a reusable experiment harness.** Set the knobs in the *Experiment config* cell,
then re-run any step. With `RUN_AWS = False` it runs end-to-end on local pandas only (no creds, no
cost); flip the guards on once you have credentials + the provisioned resources.

> Single source of truth: every step calls the shared `arbiter_rag` library — the *same code* the
> deployed agent runs. What you validate here is what ships.

## 0 · Experiment config — the only cell you edit to run a new experiment

In [1]:
# --- Experiment config (edit me) ---------------------------------------------
DATASET = "sample"          # "sample" (10 branches, data/Hawaii_Sample_Sales) or
                            # "large"  (100 branches, data/Hawaii_Electronics_100)
SALES_BUCKET = "dev-st21arbiter-poc-sales-vectors"  # S3 Vectors bucket (created by ingest script)
SALES_INDEX  = "sales-facts"                         # S3 Vectors index name
# Athena/Glue (SQL path) — the existing ARBITER structured DB + read-only workgroup.
ATHENA_DB    = "dev_st21arbiter_poc_structured"
ATHENA_WG    = "dev-st21arbiter-poc-wg"
ATHENA_OUT   = "s3://dev-st21arbiter-poc-processed/athena-results/"
TOP_K        = 3            # retrieval depth for the fuzzy path
CHUNK_VERSION = "v1"        # stamped into fact keys/metadata (re-ingest idempotency)
USE_RERANK   = False        # Bedrock rerank (needs bedrock:Rerank IAM); off by default

# Guards — keep False to run offline (local pandas only, no AWS calls, no cost).
RUN_AWS    = False          # embed + S3 Vectors ingest/query + generate_sql (Bedrock)
RUN_ATHENA = False          # execute generated SQL on the live Athena workgroup

# --- Bootstrap: import the shared library (pip install -e rag_src, or sys.path) ----
import sys, pathlib, json
_here = pathlib.Path.cwd()
_rag_src = next((p for p in [_here, *_here.parents] if (p / "arbiter_rag").is_dir()), None)
if _rag_src and str(_rag_src) not in sys.path:
    sys.path.insert(0, str(_rag_src))

import pandas as pd
import arbiter_rag
from arbiter_rag.config import get_settings, DATA_ROOT
from arbiter_rag import loaders, serialization, embeddings, vectors, retrieval, athena_sql, evaluation

S = get_settings()
DATASET_DIR = DATA_ROOT / ("Hawaii_Electronics_100" if DATASET == "large" else "Hawaii_Sample_Sales")
print("dataset dir :", DATASET_DIR, "| exists:", DATASET_DIR.exists())
print("gen model   :", S.generation_model_id)      # Amazon Nova 2 Lite (swap via settings.toml)
print("embed model :", S.embedding_model_id, f"({S.embedding_dim}d)")
print("vector index:", f"{SALES_BUCKET}/{SALES_INDEX}", "| rerank:", USE_RERANK)
print("guards      : RUN_AWS =", RUN_AWS, "| RUN_ATHENA =", RUN_ATHENA)

dataset dir : /Users/sridharpchome/Documents/Claude/Demo-ST21Arbiter/data/Hawaii_Sample_Sales | exists: True
gen model   : us.amazon.nova-2-lite-v1:0
embed model : amazon.titan-embed-text-v2:0 (1024d)
vector index: dev-st21arbiter-poc-sales-vectors/sales-facts | rerank: False
guards      : RUN_AWS = False | RUN_ATHENA = False


## 1 · Load the data

The Hawaii sales export is one CSV per branch. `loaders.load_hawaii_sales` concatenates them into a
single typed DataFrame (numeric columns coerced) so aggregation is exact.

In [2]:
sales_df = loaders.load_hawaii_sales(DATASET_DIR)
print(f"rows: {len(sales_df):,} | branches: {sales_df.branch_id.nunique()} | "
      f"islands: {sales_df.island.nunique()} | categories: {sales_df.category.nunique()}")
print(f"total revenue: ${sales_df.revenue.sum():,.2f} | total units: {int(sales_df.quantity.sum()):,}")
sales_df.head(3)

rows: 702 | branches: 10 | islands: 1 | categories: 14
total revenue: $38,921.72 | total units: 8,689


,transaction_id,branch_id,city,island,state,timestamp,sku,product_description,category,quantity,unit_cost,unit_price,revenue,cost,gross_margin,salesperson,customer_type,payment_method,inventory_remaining
0,HI001-TX-0001,HI001,Honolulu,Oahu,HI,2026-06-30 17:24:52,BAT-AAH,AA Battery Holder,Batteries,11,0.52,2.83,31.13,5.72,25.41,Keahi,Commercial,Purchase Order,751
1,HI001-TX-0002,HI001,Honolulu,Oahu,HI,2026-06-30 09:39:49,PWR-BUCK,Buck Converter Module,Power,21,1.05,4.74,99.54,22.05,77.49,Kai,Retail,Debit Card,368
2,HI001-TX-0003,HI001,Honolulu,Oahu,HI,2026-06-30 16:45:41,USB-CBO,USB-C Breakout Board,Connectors,23,1.46,5.13,117.99,33.58,84.41,Noelani,Maker/Hobbyist,Debit Card,228


## 2 · Path A — serialize structured rows into natural-language facts

Tabular rows can't be embedded directly — they must become prose. We aggregate to
**branch × category** (the data is a single day, so there's no year dimension) so each vector is a
meaningful summary, not a noisy single transaction.

In [3]:
facts = serialization.build_sales_facts(sales_df, chunking_version=CHUNK_VERSION)
print(len(facts), "sales facts (branch × category)\n")
print("example key :", facts[0]["key"])
print("example text:", facts[0]["text"])
print("filterable  :", [k for k in facts[0]["metadata"] if k not in vectors.SALES_NON_FILTERABLE_KEYS])

133 sales facts (branch × category)

example key : sales-hi001-batteries
example text: On 2026-06-30, the Honolulu branch (HI001) on Oahu sold 11 units of Batteries products across 1 transactions (1 distinct SKUs), generating $31 in revenue at a gross margin of $25 (cost of goods $6).
filterable  : ['domain', 'branch_id', 'city', 'island', 'category', 'metric_type', 'chunking_version']


### 2a · Embed + ingest into S3 Vectors  *(guarded: RUN_AWS)*

Titan v2 embeds each fact; we upsert into the immutable `sales-facts` index. Explicit bucket/index
names are passed so the notebook targets exactly the same index the deployed agent queries.

In [4]:
if RUN_AWS:
    vx = vectors.make_client(S.region)
    vectors.ensure_vector_bucket(vx, SALES_BUCKET)
    vectors.ensure_index(vx, SALES_BUCKET, SALES_INDEX, S.embedding_dim,
                         S.distance_metric, vectors.SALES_NON_FILTERABLE_KEYS)
    vecs = embeddings.embed_texts([f["text"] for f in facts], S)
    recs = [{"key": f["key"], "embedding": v, "metadata": f["metadata"]} for f, v in zip(facts, vecs)]
    n = vectors.put_records(vx, SALES_BUCKET, SALES_INDEX, recs, batch_size=S.ingest_batch_size)
    print(f"ingested {n} fact vectors into {SALES_BUCKET}/{SALES_INDEX}")
else:
    print("RUN_AWS=False — skipped ingest. Would embed + upsert", len(facts), "vectors.")

RUN_AWS=False — skipped ingest. Would embed + upsert 133 vectors.


### 2b · A works for FUZZY questions  *(guarded: RUN_AWS)*

"How did marine electronics do at the Kailua store?" is a semantic lookup — vectors handle it well.

In [5]:
FUZZY_Q = "How did marine electronics perform at the Kailua branch?"
if RUN_AWS:
    for c in retrieval.retrieve(FUZZY_Q, SALES_INDEX, S, use_rerank=USE_RERANK, top_k=TOP_K):
        d = f"{c.distance:.4f}" if c.distance is not None else "  n/a "
        print(d, " ", c.text)
else:
    print("RUN_AWS=False — would semantically retrieve top", TOP_K, "facts for:\n ", FUZZY_Q)

RUN_AWS=False — would semantically retrieve top 3 facts for:
  How did marine electronics perform at the Kailua branch?


## 3 · …but Path A FAILS at exact aggregation (the trap)

Ask for "total revenue" and the vector store returns only the top-k *individual* branch-category
facts. Summing those is **not** the company total — it's just what retrieval happened to surface.
Never answer aggregation from the semantic index.

In [6]:
real_total = sales_df.revenue.sum()
print(f"real total revenue (pandas ground truth): ${real_total:,.2f}")
if RUN_AWS:
    hits = retrieval.retrieve("What was the total revenue across all branches?",
                              SALES_INDEX, S, use_rerank=False, top_k=5)
    import re
    partial = sum(float(re.search(r"\$([\d,]+)", h.text).group(1).replace(",", ""))
                  for h in hits if re.search(r"\$([\d,]+)", h.text))
    print(f"sum of the 5 retrieved facts:            ${partial:,.0f}  <-- WRONG (only top-k)")
    print(f"the retrieved sum captures {partial/real_total:5.1%} of the true total — do not trust it.")
else:
    print("RUN_AWS=False — the live top-k-sum would be a small, misleading fraction of the true total.")

real total revenue (pandas ground truth): $38,921.72
RUN_AWS=False — the live top-k-sum would be a small, misleading fraction of the true total.


## 4 · Path B — text-to-SQL over Athena for exact aggregation

The agent generates ONE read-only SQL SELECT, we **validate** it (single-statement, SELECT-only,
table-allowlisted — a security control), then run it on a read-only Athena workgroup. Here we show
the generated (or, offline, a representative) SQL and cross-check the answer against pandas.

In [7]:
QUESTION = "What was the total revenue for each product category? Give the top 5."
if RUN_AWS:
    sql = athena_sql.generate_sql(QUESTION, S)                 # LLM → SQL (Bedrock)
else:
    sql = ("SELECT category, SUM(revenue) AS total_revenue "  # representative example, offline
           f"FROM {S.glue_table} GROUP BY category ORDER BY total_revenue DESC LIMIT 5")
print("SQL:\n", sql, "\n")
print("validation:", athena_sql.validate_sql(sql, S.glue_table)[:70], "…  (passed read-only guard)")

# Pandas ground truth for the same question — the offline oracle the eval harness reuses.
truth = (sales_df.groupby("category").revenue.sum().sort_values(ascending=False).head(5))
print("\npandas ground truth (top 5 categories by revenue):")
for cat, rev in truth.items():
    print(f"  {cat:22} ${rev:,.2f}")

SQL:
 SELECT category, SUM(revenue) AS total_revenue FROM hawaii_sales GROUP BY category ORDER BY total_revenue DESC LIMIT 5 

validation: SELECT category, SUM(revenue) AS total_revenue FROM hawaii_sales GROUP …  (passed read-only guard)

pandas ground truth (top 5 categories by revenue):
  Microcontrollers       $10,134.50
  Connectors             $6,530.84
  Power                  $3,832.22
  Sensors                $3,374.51
  Marine Electronics     $2,746.37


In [8]:
# Execution requires the deployed Glue table + Athena workgroup. Guarded:
if RUN_AWS and RUN_ATHENA:
    result = athena_sql.run_query(sql, S, database=ATHENA_DB, workgroup=ATHENA_WG,
                                  output_location=ATHENA_OUT)
    for row in result.rows:
        print(row)
    print("\nscanned bytes:", result.scanned_bytes)
else:
    print("Athena execution skipped (RUN_ATHENA=False). Cross-check against the pandas truth above.")

Athena execution skipped (RUN_ATHENA=False). Cross-check against the pandas truth above.


## 5 · The router — which tool answers which question

The agent's system prompt encodes this split. Any count/sum/avg/top-N/trend → SQL; fuzzy
"tell me about / how did X do" → semantic. The golden set pins the expected routing so a regression
(the agent picking the wrong tool) is caught by eval.

In [9]:
from arbiter_rag.config import RAG_ROOT
golden = RAG_ROOT / "eval" / "golden" / "sales_hawaii_qa.jsonl"
if golden.exists():
    cases = [json.loads(l) for l in golden.read_text().splitlines() if l.strip()]
    for c in cases:
        print(f"  [{c['expected_tool']:14}] {c['question']}")
else:
    print("golden set not found yet:", golden, "(created in Phase 4 — eval/golden/sales_hawaii_qa.jsonl)")

  [sales_sql     ] What was the total revenue across all branches?
  [sales_sql     ] How many units were sold in total across all branches?
  [sales_sql     ] What was the total gross margin across all branches?
  [sales_sql     ] How many transactions were recorded in total?
  [sales_sql     ] What was the total revenue for the Microcontrollers category?
  [sales_sql     ] How many units of Marine Electronics were sold across all branches?
  [sales_sql     ] What was the total revenue on Maui?
  [sales_sql     ] Which product category generated the most revenue, and how much?
  [sales_sql     ] Which branch sold the most units, and how many?
  [sales_sql     ] What was the average revenue per transaction?
  [sales_semantic] How did Marine Electronics perform at the Kailua branch?
  [sales_semantic] How are microcontrollers selling at the Honolulu branch?
  [sales_semantic] Tell me about Solar Power sales on Maui.


## 6 · Reuse this lab for experiments

Change one knob in the *Experiment config* cell and re-run the relevant step:

| To measure the effect of… | change | re-run |
|---|---|---|
| **more data** | `DATASET = "large"` | §1 → §2 (re-ingest) → §2b |
| **retrieval depth** | `TOP_K` | §2b, §3 |
| **reranking** | `USE_RERANK = True` (needs `bedrock:Rerank`) | §2b |
| **the generation model** | edit `generation_model_id` in `config/settings.toml` (Nova → Claude/Llama) | all AWS steps |
| **a new fact grain** | edit `serialization.build_sales_facts`, bump `CHUNK_VERSION` | §2 → re-ingest |

Then run the scaled evaluation (`python eval/run_sales_eval.py --dataset large`) to get
retrieval recall@k, routing accuracy, and numeric accuracy vs the pandas ground truth, and compare
`eval/results/` across runs. That objective loop is what drives enhancements.

## SA takeaways
- **Structured data must be serialized** before it can enter a vector store — and even then it only
  answers *fuzzy* questions.
- **Aggregation belongs in SQL.** Route numeric questions to Athena; never trust a summed top-k.
- **Text-to-SQL is an injection surface** — `validate_sql()` enforces read-only, single-statement,
  table-allowlisted queries; the Athena workgroup caps bytes scanned.
- The agent picks the tool; the golden set pins the routing so regressions are caught.